# 电子行业动量因子研究

本 notebook 是因子研究的唯一运行入口，不参与策略生成和回测。研究对象为申万电子（801080）。因子实现位于 `src/backtester/signals/momentum_factors.py`，相关性与截面回归统计位于 `src/backtester/analytics/`。

## 研究口径

1. 月末计算 13 个动量因子，并执行 $median \pm 5MAD$ 去极值、截面标准化及缺失值置零。
2. 因子相关性：每月计算股票截面的 Pearson 相关系数，汇总最近 36 个月的均值、样本标准差及 `均值/标准差`。
3. 因子回归：对每个因子分别进行月度横截面 WLS：$r_{i,T+1}^{excess}=\alpha_T+X_{i,T}f_T+\epsilon_{i,T}$。
4. 回归权重为 $\sqrt{流通市值}$；因变量是相对沪深300的下月超额收益；标准误使用经典 WLS 无偏残差方差估计。
5. 每个截面至少需要 10 只股票。当前只研究一个行业，因此不加入行业虚拟变量。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from backtester.analytics.factor_correlation import summarize_factor_correlations
from backtester.analytics.factor_regression import run_monthly_factor_regressions, summarize_factor_regressions
from backtester.analytics.factor_ic import calculate_monthly_factor_ic, summarize_factor_ic, cumulative_factor_ic
from backtester.analytics.factor_layer_backtest import prepare_layer_backtest_sample, run_quintile_backtests, summarize_layer_performance
from backtester.config import load_config_directory
from backtester.signals.momentum_factors import FACTOR_NAMES, build_momentum_factor_exposures

CONFIG = load_config_directory(PROJECT_ROOT / 'configs')['dataset']
RESULTS = PROJECT_ROOT / 'notebooks' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. 读取研究数据

In [ ]:
stocks = pd.read_csv(PROJECT_ROOT / CONFIG['raw_stock_daily_file'])
market = pd.read_csv(PROJECT_ROOT / CONFIG['raw_market_file'])
regression_sample = pd.read_csv(PROJECT_ROOT / CONFIG['output_file'])
universe = regression_sample[['formation_date', 'symbol']]
print(f"股票日频: {len(stocks):,}; 月度回归样本: {len(regression_sample):,}")

## 2. 计算并保存 13 个因子

In [ ]:
raw_exposures, processed_exposures = build_momentum_factor_exposures(
    stocks, market, universe,
    windows=tuple(CONFIG['factor_windows_months']),
    halpha_lookback=int(CONFIG['halpha_lookback_months']),
    mad_multiplier=float(CONFIG['mad_multiplier']),
)
raw_exposures.to_csv(RESULTS / 'momentum_factor_exposures_raw.csv', index=False, encoding='utf-8-sig')
processed_exposures.to_csv(RESULTS / 'momentum_factor_exposures_processed.csv', index=False, encoding='utf-8-sig')
print(f"因子暴露: {len(processed_exposures):,} 行，{processed_exposures['formation_date'].nunique()} 个月")

## 3. 因子相关性分析

In [ ]:
mean_corr, std_corr, strength, monthly_corr = summarize_factor_correlations(
    processed_exposures, FACTOR_NAMES, int(CONFIG['correlation_window_months'])
)
mean_corr.to_csv(RESULTS / 'factor_correlation_mean_last_36m.csv', encoding='utf-8-sig')
std_corr.to_csv(RESULTS / 'factor_correlation_std_last_36m.csv', encoding='utf-8-sig')
strength.to_csv(RESULTS / 'factor_correlation_strength_last_36m.csv', encoding='utf-8-sig')
monthly_corr.to_csv(RESULTS / 'monthly_factor_correlations_last_36m.csv', index=False, encoding='utf-8-sig')

### 表 1：月度相关系数序列均值

In [ ]:
mean_corr.round(2)

### 表 2：月度相关系数序列标准差

In [ ]:
std_corr.round(2)

### 表 3：因子相关强度（最近 36 个月）

In [ ]:
strength.round(2)

## 4. 月度横截面回归与 t 值

每个因子单独回归。月度回归系数是该月因子收益率，t 值检验当月因子收益是否显著异于零。

In [ ]:
monthly_regressions = run_monthly_factor_regressions(
    processed_exposures, regression_sample, FACTOR_NAMES,
    target_column='next_month_excess_return',
    weight_column='wls_weight',
    minimum_observations=10,
)
regression_summary = summarize_factor_regressions(monthly_regressions, FACTOR_NAMES)
monthly_regressions.to_csv(RESULTS / 'factor_regression_monthly.csv', index=False, encoding='utf-8-sig')
regression_summary.to_csv(RESULTS / 'factor_regression_summary.csv', index=False, encoding='utf-8-sig')

In [ ]:
regression_table = regression_summary.set_index('factor')[
    ['mean_abs_t', 'abs_t_gt_2_ratio', 'mean_t', 'mean_t_over_std',
     'mean_factor_return', 'factor_return_series_t']
].copy()
regression_table.columns = ['|t|均值', '|t|>2占比', 't均值', 't均值/t标准差', '因子收益率均值', '因子收益率序列t检验']
display_table = regression_table.copy().astype(object)
display_table['|t|均值'] = regression_table['|t|均值'].map(lambda x: f'{x:.2f}')
display_table['|t|>2占比'] = regression_table['|t|>2占比'].map(lambda x: f'{x:.2%}')
display_table['t均值'] = regression_table['t均值'].map(lambda x: f'{x:.2f}')
display_table['t均值/t标准差'] = regression_table['t均值/t标准差'].map(lambda x: f'{x:.2f}')
display_table['因子收益率均值'] = regression_table['因子收益率均值'].map(lambda x: f'{x:.3%}')
display_table['因子收益率序列t检验'] = regression_table['因子收益率序列t检验'].map(lambda x: f'{x:.2f}')
display_table

## 5. 因子 IC 研究

每个月计算当月因子暴露与下月个股收益的横截面 Pearson 相关系数。由于每个月的沪深300收益对所有股票相同，使用个股原始收益或超额收益得到的 Pearson IC 完全相同。IR 按参考表定义为 $|mean(IC)|/std(IC)$。

In [ ]:
monthly_ic = calculate_monthly_factor_ic(
    processed_exposures, regression_sample, FACTOR_NAMES,
    return_column='next_month_return', minimum_observations=10,
)
ic_summary = summarize_factor_ic(monthly_ic, FACTOR_NAMES)
cumulative_ic = cumulative_factor_ic(monthly_ic, FACTOR_NAMES)
monthly_ic.to_csv(RESULTS / 'factor_ic_monthly.csv', index=False, encoding='utf-8-sig')
ic_summary.to_csv(RESULTS / 'factor_ic_summary.csv', index=False, encoding='utf-8-sig')
cumulative_ic.to_csv(RESULTS / 'factor_ic_cumulative.csv', encoding='utf-8-sig')

In [ ]:
ic_table = ic_summary.set_index('factor')[
    ['mean_ic', 'std_ic', 'ir', 'positive_ic_ratio', 'abs_ic_gt_0_02_ratio']
].copy()
ic_table.columns = ['IC序列均值', 'IC序列标准差', 'IR比率', 'IC>0占比', '|IC|>0.02占比']
ic_display = ic_table.copy().astype(object)
for column in ['IC序列均值', 'IC序列标准差', 'IC>0占比', '|IC|>0.02占比']:
    ic_display[column] = ic_table[column].map(lambda x: f'{x:.2%}')
ic_display['IR比率'] = ic_table['IR比率'].map(lambda x: f'{x:.2f}')
ic_display

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
ax = cumulative_ic.plot(figsize=(13, 7), linewidth=1.4)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('电子行业动量因子累计 IC')
ax.set_xlabel('截面日期')
ax.set_ylabel('累计 IC')
ax.legend(ncol=2, fontsize=8)
ax.figure.tight_layout()
ax.figure.savefig(RESULTS / 'factor_ic_cumulative.png', dpi=160, bbox_inches='tight')
plt.show()

## 6. 五层组合回测

每月把因子从小到大分为五组，组1是最低暴露、组5是最高暴露。所有组合在电子行业内等权；基准组合为当月全部合格股票的等权组合；多空组合做多组1、做空组5。因子于月末计算，在下月首个交易日收盘建仓并持有到下月末收盘。本研究回测暂不计交易成本。

In [ ]:
benchmark_daily = pd.read_csv(PROJECT_ROOT / CONFIG['raw_benchmark_file'])
layer_sample = prepare_layer_backtest_sample(
    processed_exposures, regression_sample, stocks, benchmark_daily, FACTOR_NAMES
)
layer_monthly = run_quintile_backtests(layer_sample, FACTOR_NAMES, quantiles=5)
layer_performance = summarize_layer_performance(layer_monthly)
LAYER_RESULTS = RESULTS / 'layer_backtest'
LAYER_RESULTS.mkdir(parents=True, exist_ok=True)
layer_monthly.to_csv(LAYER_RESULTS / 'monthly_returns_all_factors.csv', index=False, encoding='utf-8-sig')
layer_performance.to_csv(LAYER_RESULTS / 'performance_all_factors.csv', index=False, encoding='utf-8-sig')
print(f"分层月收益记录: {len(layer_monthly):,}; 绩效记录: {len(layer_performance):,}")

In [ ]:
portfolio_labels = {
    'group_1': '组合1', 'group_2': '组合2', 'group_3': '组合3',
    'group_4': '组合4', 'group_5': '组合5', 'baseline': '等权基准',
    'long_short': '多空组合', 'hs300': '沪深300',
}
metric_columns = [
    'annualized_return', 'annualized_volatility', 'sharpe_ratio', 'max_drawdown',
    'annualized_excess_return', 'excess_annualized_volatility', 'information_ratio',
    'relative_monthly_win_ratio', 'excess_max_drawdown',
]
metric_labels = [
    '年化收益率', '年化波动率', '夏普比率', '最大回撤',
    '年化超额收益率', '超额年化波动率', '信息比率', '相对基准月胜率', '超额最大回撤',
]
report_links = []
for factor in FACTOR_NAMES:
    factor_dir = LAYER_RESULTS / factor
    factor_dir.mkdir(parents=True, exist_ok=True)
    monthly = layer_monthly[layer_monthly['factor'] == factor].copy()
    performance = layer_performance[layer_performance['factor'] == factor].copy()
    monthly.to_csv(factor_dir / 'monthly_returns.csv', index=False, encoding='utf-8-sig')
    performance.to_csv(factor_dir / 'performance.csv', index=False, encoding='utf-8-sig')
    pivot = monthly.pivot(index='label_end_date', columns='portfolio', values='return').sort_index()
    pivot.index = pd.to_datetime(pivot.index)
    net = (1.0 + pivot).cumprod()
    relative_net = net[[f'group_{i}' for i in range(1, 6)]].div(net['baseline'], axis=0)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    net[[f'group_{i}' for i in range(1, 6)] + ['baseline', 'hs300']].rename(columns=portfolio_labels).plot(ax=axes[0, 0])
    axes[0, 0].set_title(f'{factor}: Layer Portfolio Net Values')
    axes[0, 0].set_ylabel('Net value')
    relative_net.rename(columns=portfolio_labels).plot(ax=axes[0, 1])
    axes[0, 1].axhline(1.0, color='black', linewidth=0.8)
    axes[0, 1].set_title(f'{factor}: Net Values Relative to Equal-weight Baseline')
    axes[0, 1].set_ylabel('Relative net value')
    (pivot['group_1'] - pivot['hs300']).hist(ax=axes[1, 0], bins=20, color='#b40000')
    axes[1, 0].set_title(f'{factor}: Group 1 Monthly Excess Return vs HS300')
    axes[1, 0].set_xlabel('Monthly excess return')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 1].bar(pivot.index, pivot['long_short'], width=20, color='#b40000', alpha=0.65, label='Monthly return')
    axes[1, 1].axhline(0, color='black', linewidth=0.8)
    twin = axes[1, 1].twinx()
    ((1.0 + pivot['long_short']).cumprod() - 1.0).plot(ax=twin, color='#f0a000', linewidth=2, label='Cumulative return')
    axes[1, 1].set_title(f'{factor}: Low-minus-High Long-Short Portfolio')
    axes[1, 1].set_ylabel('Monthly return')
    twin.set_ylabel('Cumulative return')
    fig.tight_layout()
    fig.savefig(factor_dir / 'overview.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    table = performance.set_index('portfolio').reindex([f'group_{i}' for i in range(1, 6)] + ['baseline', 'long_short', 'hs300'])
    table = table[metric_columns]
    table.index = [portfolio_labels[index] for index in table.index]
    table.columns = metric_labels
    displayed = table.copy().astype(object)
    percent_columns = ['年化收益率', '年化波动率', '最大回撤', '年化超额收益率', '超额年化波动率', '相对基准月胜率', '超额最大回撤']
    for column in displayed.columns:
        if column in percent_columns:
            displayed[column] = table[column].map(lambda x: '' if pd.isna(x) else f'{x:.2%}')
        else:
            displayed[column] = table[column].map(lambda x: '' if pd.isna(x) else f'{x:.2f}')
    report = f"""<!doctype html><html lang='zh-CN'><head><meta charset='utf-8'><title>{factor} 分层回测</title>
    <style>body{{font-family:Arial,'Microsoft YaHei',sans-serif;margin:28px}}table{{border-collapse:collapse;font-size:12px}}th,td{{border-bottom:1px solid #ddd;padding:6px;text-align:right}}th{{border-top:2px solid #e66;border-bottom:2px solid #e66}}img{{max-width:100%;margin-top:24px}}</style></head>
    <body><h1>{factor} 因子五层组合回测</h1><p>组1至组5按因子从小到大排列；行业内等权；多空组合为组1减组5；不计交易成本。</p>
    {displayed.to_html(escape=False, border=0)}<img src='overview.png' alt='绩效图'></body></html>"""
    (factor_dir / 'report.html').write_text(report, encoding='utf-8')
    report_links.append(f"<li><a href='{factor}/report.html'>{factor}</a></li>")

index_html = "<!doctype html><html lang='zh-CN'><head><meta charset='utf-8'><title>动量因子分层回测</title></head><body><h1>13个动量因子分层回测</h1><ul>" + ''.join(report_links) + "</ul></body></html>"
(LAYER_RESULTS / 'index.html').write_text(index_html, encoding='utf-8')

In [ ]:
layer_performance[layer_performance['factor'] == 'HAlpha'][
    ['portfolio'] + metric_columns
].replace({'portfolio': portfolio_labels})

## 数据限制

免费申万接口没有完整历史调入调出记录，因此当前成分股回溯存在幸存者偏差；BaoStock 没有独立可靠的历史 PT 字段。回归、IC 与分层回测结果适合验证研究流程与比较因子，不应直接解释为无偏的全市场历史结论。